# SFINCS model build – San Felipe (Yucatán, Mexico)

This notebook builds a quadtree [SFINCS](https://sfincs.readthedocs.io) flood model for San Felipe with [HydroMT-SFINCS](https://deltares.github.io/hydromt_sfincs/). Running it from top to bottom writes a complete, ready-to-run model to `sfincs_san_felipe_qtr/` (an existing folder is overwritten). Running the model and post-processing are not part of this notebook.

## Notebook outline

| Section | Content |
|---|---|
| 1. Imports and paths | Input files (`data/`), global parameters, file check |
| 2. Initialise model | Model object, simulation period, initial water level |
| 3. Grid | Rotated quadtree grid, refined over the LiDAR extent |
| 4. Elevation | LiDAR with bathymetry as fallback |
| 5. Mask | Active cells and offshore water level boundary |
| 6. Structures | Buildings represented as raised grid cells |
| 7. Observation points | Output points for time series |
| 8. Manning roughness and Infiltration| From ESA WorldCover |
| 9. Boundary conditions | Water levels from GTSM (NetCDF) or El Cuyo (CSV) |
| 10. Forcings | 10.1 wind, 10.2 precipitation |
| 11. Writing files | Output interval and writing the model to disk |

## How to use

1. Put the input files listed in section 1 in the `data/` folder.
2. Check the parameters in section 1 (domain, resolution, boundary condition source, simulation period).
3. Run all cells in order. Sections depend on each other, so use *Restart kernel and run all* when rebuilding.

All model data are in **EPSG:32616** (UTM zone 16N). All times are in **UTC**.

## 1. Imports and paths

All input files are read from the `data/` folder and defined once in the path cell below, so a renamed or moved file only has to be changed there.

| File in `data/` | Section | Description |
|---|---|---|
| `Lidar_gecorrigeerd_SanFelipe.tif` | 3, 4 | Corrected, gap-free LiDAR DEM (UTM 16N). First elevation source; its extent defines the refinement area |
| `gecombineerde_bathymetrie.tif` | 4 | Combined bathymetry/topography raster; fills everything the LiDAR does not cover |
| `SanFelipe_buildings.gpkg` | 6 | Building footprints |
| `street_points.csv` | 7 | Observation points (columns `Name`, `X` = longitude, `Y` = latitude, WGS84) |
| `ESA_WorldCover_10m_2021_v200_N21W090_Map.tif` | 8, 10.3 | ESA WorldCover 2021 land cover (10 m) |
| `reanalysis_waterlevel_10min_2024_02_v3.nc` | 9.1 | GTSM reanalysis water levels, February 2024 (Norte event), 10-minute |
| `currents_SBT_WD_mcspd_mcdir_depth_u_v_w_202510_awac_elCuyo.csv` | 9.2 | AWAC measurements at El Cuyo (offshore), used for the validation run |
| `era5wind_2021_2026.grib` | 10.1 | ERA5 10 m wind (`u10`, `v10`) |
| `precipitation14_26.grib` | 10.2 | ERA5 total precipitation (`tp`) |

Only one of the two water level files is needed, depending on `bc_source`. `worldcover_manning.csv` and `worldcover_infiltration.csv` are generated by sections 8 and 10.3 and written to `data/`.

`hydromt` is imported only to register the `.raster` accessor that is used on the forcing datasets, so it should not be removed.

In [ ]:
import os
from pathlib import Path

import geopandas as gpd
import hydromt
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio
import xarray as xr
from hydromt_sfincs import SfincsModel
from pyproj import Transformer
from shapely import affinity
from shapely.geometry import LineString, Point, box

In [ ]:
data_dir = Path("data")

lidar_tif = data_dir / "Lidar_gecorrigeerd_SanFelipe.tif"
topography_tif = data_dir / "gecombineerde_bathymetrie.tif"
buildings_gpkg = data_dir / "SanFelipe_buildings.gpkg"
street_points_csv = data_dir / "street_points.csv"
worldcover_tif = data_dir / "ESA_WorldCover_10m_2021_v200_N21W090_Map.tif"
gtsm_nc = data_dir / "reanalysis_waterlevel_10min_2024_02_v3.nc"
el_cuyo_csv = data_dir / "currents_SBT_WD_mcspd_mcdir_depth_u_v_w_202510_awac_elCuyo.csv"
wind_grib = data_dir / "era5wind_2021_2026.grib"
precip_grib = data_dir / "precipitation14_26.grib"

manning_csv = data_dir / "worldcover_manning.csv"
infiltration_csv = data_dir / "worldcover_infiltration.csv"

model_root = Path("sfincs_san_felipe_qtr")

### Parameters

`bc_source` selects the water level boundary condition and with it the simulation period:

- `"gtsm"`: GTSM reanalysis (NetCDF), Norte event of 4–7 February 2024
- `"el_cuyo"`: El Cuyo AWAC measurements (CSV), 8–11 November 2025 (validation period)

Mean sea level is not 0 in the elevation datum of the model but at `msl_offset` = −0.5 m. This value is used as initial water level and as offset on the boundary water levels.

In [ ]:
bc_source = "gtsm"  # water level boundary: "gtsm" (NetCDF) or "el_cuyo" (CSV)
periods = {
    "gtsm": ("20240204 000000", "20240207 000000"),
    "el_cuyo": ("20251108 000000", "20251111 000000"),
}
tstart, tstop = periods[bc_source]  # simulation period, UTC (YYYYMMDD HHMMSS)

target_crs = "EPSG:32616"  # model CRS (UTM zone 16N)
lat_min, lat_max = 21.562176, 21.607176  # domain extent, WGS84 [deg]
lon_min, lon_max = -88.251847, -88.181847
rotation = 35.0  # rotation of the domain around its centre [deg, counter-clockwise]
res = 40.0  # coarsest cell size [m]
refine_level = 3  # refinement levels inside the LiDAR extent, each level halves the cell size

msl_offset = -0.5  # mean sea level in the model elevation datum [m]
time_margin = pd.Timedelta(hours=1)  # margin around the model period when cutting forcing data
era5_pad = 0.5  # margin around the model domain when cutting ERA5 data [deg]

required = [
    lidar_tif, topography_tif, buildings_gpkg, street_points_csv, worldcover_tif, wind_grib, precip_grib,
    gtsm_nc if bc_source == "gtsm" else el_cuyo_csv,
]
missing = [str(p) for p in required if not p.exists()]
assert not missing, f"Missing input files: {missing}"

model_root.mkdir(exist_ok=True)

## 2. Initialise model

Creates the SFINCS model object in write mode (`w+`) in `model_root`; an existing model in that folder is overwritten. `write_gis=True` also writes GIS files of the model layers.

The config sets the quadtree grid file, switches SnapWave off and defines the simulation period (`tref`, `tstart`, `tstop`). Advection and Coriolis are switched off.

In [ ]:
sf = SfincsModel(root=model_root, mode="w+", write_gis=True)
sf.config.update({"snapwave": 0, "qtrfile": "sfincs.qtr"})
sf.config.update_grid_from_config()

sf.config.update({
    "tref": tstart,
    "tstart": tstart,
    "tstop": tstop,
    "zsini": msl_offset,
    "advection": 0,  # 0 = off
    "coriolis": 0,  # 0 = off
})

## 3. Grid

The model domain is defined as a lat/lon box (`lat_min` … `lon_max`), converted to UTM and rotated by `rotation` degrees around its centre.

The quadtree grid starts with cells of `res` metres and is refined inside the extent of the LiDAR raster. Each refinement level halves the cell size (`refine_level = 3`: 40 m → 5 m). The printed cell sizes confirm the result. The LiDAR raster must have a CRS equal to `target_crs`.

In [ ]:
t = Transformer.from_crs("EPSG:4326", target_crs, always_xy=True)
x_min, y_min = t.transform(lon_min, lat_min)
x_max, y_max = t.transform(lon_max, lat_max)
x_c, y_c = (x_min + x_max) / 2, (y_min + y_max) / 2

domain_rot = affinity.rotate(box(x_min, y_min, x_max, y_max), rotation, origin=(x_c, y_c))
region_gdf = gpd.GeoDataFrame(geometry=[domain_rot], crs=target_crs)

with rasterio.open(lidar_tif) as src:
    assert src.crs is not None, "LiDAR raster has no CRS"
    lidar_bounds = src.bounds

refinement_polygons = gpd.GeoDataFrame(
    {"refinement_level": [refine_level]},
    geometry=[box(*lidar_bounds)],
    crs=target_crs,
)

sf.quadtree_grid.create_from_region(
    region={"geom": region_gdf},
    res=res,
    crs=target_crs,
    rotated=True,
    refinement_polygons=refinement_polygons,
)

cell_size = np.sqrt(np.asarray(sf.quadtree_grid.data.ugrid.grid.area))
print(f"Cells: {cell_size.size} | cell size min / max: {cell_size.min():.1f} / {cell_size.max():.1f} m")

## 4. Elevation

Elevation is interpolated onto the quadtree cells from a prioritised list of rasters: the first raster has priority and the following ones only fill cells without data. The LiDAR comes first and the coarser bathymetry fills everything outside the LiDAR coverage. `buffer_cells=1` reads the rasters slightly beyond the domain to avoid edge effects.

The printed check shows the elevation range and the number of cells without data (NaN).

In [ ]:
elevation_list = [
    {"elevation": os.path.abspath(lidar_tif)},
    {"elevation": os.path.abspath(topography_tif)},
]

sf.quadtree_elevation.create(elevation_list=elevation_list, buffer_cells=1)
sf.quadtree_elevation.write()

z = sf.quadtree_elevation.data["z"]
print(f"z min / mean / max: {float(z.min()):.2f} / {float(z.mean()):.2f} / {float(z.max()):.2f} m")
print(f"Cells without data (NaN): {int(np.isnan(z.values).sum())} of {z.size}")

## 5. Mask

The mask defines the role of every cell: `0` inactive, `1` active, `2` water level boundary.

- **Active cells:** all cells inside the domain with an elevation above `zmin`. With `zmin = -50` m this is the whole domain (the deepest point is about −12.7 m).
- **Boundary cells:** cells below `boundary_zmax` inside a strip of `strip_width` metres along the offshore edge of the domain (north-west side after rotation). The strip is built on the unrotated domain and then rotated with the same angle and origin as the domain.

In [ ]:
zmin = -50.0  # cells below this elevation are inactive [m]
boundary_zmax = 0.0  # only cells below this elevation can become boundary cells [m]
strip_width = 100.0  # width of the boundary strip along the offshore edge [m]

sf.quadtree_mask.create_active(zmin=zmin, include_polygon=region_gdf, reset_mask=True)
mask = sf.quadtree_grid.data["mask"]
n_active = int((mask.values == 1).sum())
assert n_active > 0, "No active cells"
print(f"Active cells: {n_active} of {mask.size}")

strip = box(x_min, y_max - strip_width, x_max, y_max + 2 * res)
strip_gdf = gpd.GeoDataFrame(geometry=[affinity.rotate(strip, rotation, origin=(x_c, y_c))], crs=target_crs)

sf.quadtree_mask.create_boundary(
    btype="waterlevel",
    zmax=boundary_zmax,
    include_polygon=strip_gdf,
    reset_bounds=True,
)
sf.quadtree_mask.write()

### Quick visualisation of the domain

Satellite basemap with the model region (needs an internet connection), followed by the elevation on the quadtree grid.

In [ ]:
sf.plot_basemap(
    variable="mask",
    shaded=True,
    plot_region=True,
    plot_bounds=False,
    bmap="sat",
    zoomlevel=13,
    figsize=(12, 8),
    alpha=0.0,
)
plt.show()

fig, ax = plt.subplots(figsize=(12, 8))
sf.quadtree_elevation.data["z"].ugrid.plot(ax=ax, cmap="terrain", vmin=-2, vmax=5, edgecolors="none")
ax.set_aspect("equal")
ax.set_title("Elevation and quadtree refinement")
plt.show()

## 6. Structures

Buildings are added as raised cells: every grid cell whose centre lies inside a selected building is raised by `building_raise` metres, so that the buildings act as obstacles to the flow.

Only buildings above a slanted cut-off line are used, above this line the LIDAR data captured the buildings already. 

- `y_offset`: shifts the line up (+) or down (−) [m]
- `line_angle`: angle of the line [degrees]

The elevation is re-created from the rasters before the cells are raised, so re-running this cell never adds the increase twice. The plot shows the result together with the cut-off line.

In [ ]:
y_offset = -190  # shift of the cut-off line relative to the centre of the buildings [m]
line_angle = 167.0  # angle of the cut-off line [deg]
building_raise = 3.5  # elevation increase of cells under selected buildings [m]

buildings_all = gpd.read_file(buildings_gpkg).to_crs(target_crs)

minx_b, miny_b, maxx_b, maxy_b = buildings_all.total_bounds
center_x, center_y = (minx_b + maxx_b) / 2, (miny_b + maxy_b) / 2
target_y = center_y + y_offset

unrotated = buildings_all.geometry.apply(
    lambda geom: affinity.rotate(geom, -line_angle, origin=(center_x, center_y))
)
buildings_gdf = buildings_all[unrotated.centroid.y > target_y].copy()

sf.quadtree_elevation.create(elevation_list=elevation_list, buffer_cells=1)
z = sf.quadtree_elevation.data["z"]

x_faces, y_faces = z.ugrid.grid.face_x, z.ugrid.grid.face_y
grid_gdf = gpd.GeoDataFrame(
    {"cell_index": np.arange(len(x_faces))},
    geometry=gpd.points_from_xy(x_faces, y_faces),
    crs=target_crs,
)
joined = gpd.sjoin(grid_gdf, buildings_gdf, how="inner", predicate="intersects")
building_cells = joined["cell_index"].unique()

z.values[building_cells] += building_raise
sf.quadtree_elevation.data["z"] = z
print(f"Buildings selected: {len(buildings_gdf)} of {len(buildings_all)} | cells raised: {len(building_cells)}")

fig, ax = plt.subplots(figsize=(10, 10))
z.ugrid.plot(ax=ax, cmap="terrain", vmin=-3, vmax=9, edgecolors="none")
cutoff = affinity.rotate(
    LineString([(minx_b - 500, target_y), (maxx_b + 500, target_y)]),
    line_angle,
    origin=(center_x, center_y),
)
ax.plot(*cutoff.xy, "r--", lw=2, label="Cut-off line")
ax.set_xlim(minx_b - 400, maxx_b + 400)
ax.set_ylim(miny_b - 400, maxy_b + 400)
ax.set_aspect("equal")
ax.set_title("San Felipe: cut-off line and raised buildings")
ax.legend(loc="upper right")
plt.show()

## 7. Observation points

Output points at which SFINCS writes time series (`sfincs_his.nc`) when the model is run. The points are the manual points defined in the cell below plus all points in `street_points.csv`. All coordinates are lon/lat (WGS84) and are converted to the model CRS.

In [ ]:
manual_points = {  # name: (longitude, latitude)
    "Main street": (-88.232373, 21.565493),
    "Station_SF": (-88.215227, 21.572000),
}

obs_manual = gpd.GeoDataFrame(
    {"name": list(manual_points)},
    geometry=[Point(xy) for xy in manual_points.values()],
    crs="EPSG:4326",
)

df_csv = pd.read_csv(street_points_csv)
obs_csv = gpd.GeoDataFrame(
    {"name": df_csv["Name"]},
    geometry=gpd.points_from_xy(df_csv["X"], df_csv["Y"]),
    crs="EPSG:4326",
)

obs_gdf = pd.concat([obs_manual, obs_csv], ignore_index=True).to_crs(target_crs)
sf.observation_points.set(obs_gdf)
print(f"Observation points: {len(obs_gdf)}")

fig, ax = plt.subplots(figsize=(10, 8))
z.ugrid.plot(ax=ax, cmap="terrain", vmin=-3, vmax=5, edgecolors="none")
obs_gdf.plot(ax=ax, color="red", marker="*", markersize=150, edgecolor="white", label="Observation points")
ax.set_aspect("equal")
ax.set_title("Elevation and observation points")
ax.legend(loc="upper right")
plt.show()

## 8 Roughness and Infiltration

### 8.1 Manning roughness

Manning's *n* is assigned from the ESA WorldCover land cover map with a reclass table (`lulc` class → `N`). The table is defined in the cell below, written to `worldcover_manning.csv` and read by HydroMT-SFINCS.

| WorldCover class | N [s/m^(1/3)] |
|---|---|
| 10 Tree cover | 0.12 |
| 20 Shrubland | 0.05 |
| 30 Grassland | 0.034 |
| 40 Cropland | 0.037 |
| 50 Built-up | 0.1 |
| 60 Bare / sparse vegetation | 0.023 |
| 80 Permanent water | 0.02 |
| 90 Herbaceous wetland | 0.035 |
| 95 Mangroves | 0.07 |
| 100 Moss and lichen | 0.025 |

In [ ]:
manning = pd.DataFrame({
    "lulc": [10, 20, 30, 40, 50, 60, 80, 90, 95, 100],
    "N": [0.12, 0.05, 0.034, 0.037, 0.1, 0.023, 0.02, 0.035, 0.07, 0.025],
}).set_index("lulc")
manning.to_csv(manning_csv)

roughness_list = [{
    "lulc": os.path.abspath(worldcover_tif),
    "reclass_table": os.path.abspath(manning_csv),
}]
sf.quadtree_roughness.create(roughness_list=roughness_list)
sf.quadtree_roughness.write()

fig, ax = plt.subplots(figsize=(10, 8))
sf.quadtree_roughness.data["manning"].ugrid.plot(
    ax=ax, cmap="tab10", edgecolors="none", cbar_kwargs={"label": "Manning n [s/m^(1/3)]"}
)
ax.set_aspect("equal")
ax.set_title("Manning roughness")
plt.show()

### 8.2 Infiltration

Spatially varying, constant-in-time infiltration rate `qinf` [mm/h] from the same WorldCover map, using a reclass table (`lulc` class → `qinf`, written to `worldcover_infiltration.csv`). The land cover is aggregated to the model cells with `reproj_method="average"`.

| WorldCover class | qinf [mm/h] |
|---|---|
| 10 Tree cover | 50 |
| 20 Shrubland | 100 |
| 30 Grassland | 50 |
| 40 Cropland | 20 |
| 50 Built-up | 5 |
| 60 Bare / sparse vegetation | 1994.4 |
| 80 Permanent water | 0 |
| 90 Herbaceous wetland | 0.072 |
| 95 Mangroves | 0.072 |
| 100 Moss and lichen | 0 |

In [ ]:
infiltration = pd.DataFrame({
    "lulc": [10, 20, 30, 40, 50, 60, 80, 90, 95, 100],
    "qinf": [50, 100, 50, 20, 5, 1994.4, 0, 0.072, 0.072, 0],
}).set_index("lulc")
infiltration.to_csv(infiltration_csv)

sf.quadtree_infiltration.create_constant(
    lulc=os.path.abspath(worldcover_tif),
    reclass_table=os.path.abspath(infiltration_csv),
    reproj_method="average",
)
sf.quadtree_infiltration.write()

## 9. Boundary conditions

Water levels along the offshore boundary (mask value 2) come from one of two sources, selected with `bc_source` in section 1. Only the cell of the selected source runs. In both cases the data are cut to the model period plus `time_margin`, shifted by `msl_offset` and replace any water levels set earlier (`merge=False`).

The next cell derives this time window. It is also used for wind and precipitation.

In [ ]:
t0, t1 = (pd.Timestamp(t) for t in sf.get_model_time())
t0, t1 = t0 - time_margin, t1 + time_margin

### 9.1 GTSM reanalysis (NetCDF)

Water levels from the GTSM reanalysis for the Norte event of February 2024 (downloaded NetCDF, 10-minute output). The `n_stations` stations nearest to (`lon0`, `lat0`) are selected (distance in degrees) and passed to SFINCS as boundary points. A constant wave set-up of `wave_setup` metres, computed separately, is added to the GTSM water level. `buffer` is the maximum distance in metres between a station and the model region.

The cell stops if the file does not cover the model period.

In [ ]:
if bc_source == "gtsm":
    lon0, lat0 = -88.22, 21.58  # point around which the nearest stations are selected (centre of the domain)
    n_stations = 5  # number of nearest GTSM stations
    wave_setup = 0.34  # wave set-up added to the GTSM water level [m]
    buffer = 100_000  # maximum distance between a station and the model region [m]

    ds = xr.open_dataset(gtsm_nc)
    assert ds["time"].values[0] <= np.datetime64(t0) and ds["time"].values[-1] >= np.datetime64(t1), \
        "GTSM file does not cover the model period"

    dist = np.hypot(ds["station_x_coordinate"] - lon0, ds["station_y_coordinate"] - lat0)
    idx = dist.values.argsort()[:n_stations]
    sub = ds.isel(stations=idx).sel(time=slice(t0, t1)).load()

    sub = sub.rename({
        "stations": "index",
        "station_x_coordinate": "lon",
        "station_y_coordinate": "lat",
    }).set_coords(["lon", "lat"])
    if "bzs" in sub.variables:
        sub = sub.rename({"bzs": "waterlevel"})
    sub["waterlevel"] = sub["waterlevel"] + wave_setup

    sf.water_level.create(geodataset=sub, offset=msl_offset, buffer=buffer, merge=False)

### 9.2 El Cuyo measurements (CSV)

Offshore AWAC measurements at El Cuyo, used for the validation run of November 2025. The mean of the water depth is subtracted, so the series is the variation around its mean (taken as mean sea level). Date and time columns are in UTC. The series is attached to one point at the centre of the model region and therefore applies to the whole boundary.

The cell stops if the file does not cover the model period.

In [ ]:
if bc_source == "el_cuyo":
    df_cuyo = pd.read_csv(el_cuyo_csv)
    time_str = df_cuyo["Time(UTC)"].astype(int).astype(str).str.zfill(6)
    time_cuyo = pd.to_datetime(
        df_cuyo["Date (UTC)"].astype(str) + time_str.replace("240000", "000000"),
        format="%Y%m%d%H%M%S",
    )
    waterlevel = df_cuyo["water_depth(m)"] - df_cuyo["water_depth(m)"].mean()

    ts = pd.Series(waterlevel.values, index=time_cuyo, name="waterlevel").dropna().sort_index()
    ts = ts[~ts.index.duplicated(keep="first")]
    assert ts.index[0] <= t0 and ts.index[-1] >= t1, "El Cuyo file does not cover the model period"
    ts = ts.loc[t0:t1]

    centroid = sf.region.to_crs(4326).geometry.union_all().centroid
    locations = gpd.GeoDataFrame({"index": [1]}, geometry=[centroid], crs=4326).set_index("index")

    sf.water_level.create(timeseries=ts.to_frame(name=1), locations=locations, offset=msl_offset, merge=False)

## 10. Forcings

Wind and precipitation come from ERA5 GRIB files that are read with `cfgrib`. In both cases the data are cut to the model period (plus `time_margin`) and to the model domain plus `era5_pad` degrees, converted to a regular dataset with a time axis and added to the model in WGS84.

### 10.1 Wind

The 10 m wind components `u10` and `v10` [m/s] are added as spatially varying wind (`wind10_u`, `wind10_v`). SFINCS stores them in `sfincs_netamuv.nc`.

In [ ]:
ds = xr.open_dataset(wind_grib, engine="cfgrib")
tname = "valid_time" if "valid_time" in ds.coords else "time"
tdim = ds[tname].dims[0]

in_period = ((ds[tname] >= np.datetime64(t0)) & (ds[tname] <= np.datetime64(t1))).values
assert in_period.any(), "Wind file does not cover the model period"
ds = ds.isel({tdim: np.where(in_period)[0]})

minx, miny, maxx, maxy = region_gdf.to_crs(4326).total_bounds
lon = np.where(ds["longitude"].values > 180, ds["longitude"].values - 360, ds["longitude"].values)
lat = ds["latitude"].values
sel_lon = np.where((lon >= minx - era5_pad) & (lon <= maxx + era5_pad))[0]
sel_lat = np.where((lat >= miny - era5_pad) & (lat <= maxy + era5_pad))[0]
assert len(sel_lat) > 1 and len(sel_lon) > 1, "Model domain lies outside the wind file"
ds = ds.isel(latitude=sel_lat, longitude=sel_lon).load()

wind = xr.Dataset(
    {
        "wind10_u": (("time", "y", "x"), ds["u10"].transpose(tdim, "latitude", "longitude").values),
        "wind10_v": (("time", "y", "x"), ds["v10"].transpose(tdim, "latitude", "longitude").values),
    },
    coords={"time": ds[tname].values, "y": ds["latitude"].values, "x": lon[sel_lon]},
)
wind.raster.set_crs(4326)

sf.wind.create(wind=wind)
ds.close()

### 10.2 Precipitation

The ERA5 precipitation file has a forecast structure (`time` + `step`). It is flattened to one time axis using `valid_time`, duplicate times are dropped (first occurrence is kept), values are converted from m to mm and treated as hourly values (mm/h).

The first time selection is wider than the model period (2 days before, 1 day after) because `time` is the forecast start time and not the valid time; the exact cut on `valid_time` follows after flattening.

In [ ]:
ds = xr.open_dataset(precip_grib, engine="cfgrib")
da = ds["tp"]

time_dim = "time" if "time" in da.dims else "valid_time"
da = da.sel({time_dim: slice(t0 - pd.Timedelta(days=2), t1 + pd.Timedelta(days=1))})

minx, miny, maxx, maxy = region_gdf.to_crs(4326).total_bounds
lon = np.where(da["longitude"].values > 180, da["longitude"].values - 360, da["longitude"].values)
lat = da["latitude"].values
sel_lon = np.where((lon >= minx - era5_pad) & (lon <= maxx + era5_pad))[0]
sel_lat = np.where((lat >= miny - era5_pad) & (lat <= maxy + era5_pad))[0]
assert len(sel_lat) > 1 and len(sel_lon) > 1, "Model domain lies outside the precipitation file"
da = da.isel(latitude=sel_lat, longitude=sel_lon).load()

extra_dims = [d for d in da.dims if d not in ("latitude", "longitude")]
da = da.transpose(*extra_dims, "latitude", "longitude")
vals = da.values.reshape(-1, da.sizes["latitude"], da.sizes["longitude"]) * 1000.0

valid_time = da["valid_time"]
valid_time = valid_time.transpose(*[d for d in extra_dims if d in valid_time.dims]).values.reshape(-1)
assert valid_time.size == vals.shape[0], "valid_time and precipitation have different shapes"

times, first = np.unique(valid_time, return_index=True)
vals = vals[first]
keep = (times >= np.datetime64(t0)) & (times <= np.datetime64(t1))
assert keep.any(), "Precipitation file does not cover the model period"

precip = xr.Dataset(
    {"precip": (("time", "y", "x"), np.nan_to_num(vals[keep]))},
    coords={"time": times[keep], "y": da["latitude"].values, "x": lon[sel_lon]},
)
precip["precip"].attrs["units"] = "mm/h"
precip.raster.set_crs(4326)

sf.precipitation.create(precip=precip)

## 11. Writing files

Sets the interval of the map output (`dtmapout`, in seconds) and writes all model files to `model_root`. The model is now ready to be run with SFINCS.

In [ ]:
sf.config.update({"dtmapout": 900})  # interval of map output [s]
sf.write()
print(f"Model written to {model_root}")